In [1]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import plotly.express as px
from datetime import datetime

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
play_store_df = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Desktop\elevance skills project\Play Store Data.csv")

In [4]:
play_store_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
reviews_df = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Desktop\elevance skills project\User Reviews.csv")
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [6]:
play_store_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='str')

In [7]:
reviews_df.columns

Index(['App', 'Translated_Review', 'Sentiment', 'Sentiment_Polarity',
       'Sentiment_Subjectivity'],
      dtype='str')

In [8]:
play_store_df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [9]:
reviews_df.isnull().sum()

App                           0
Translated_Review         26868
Sentiment                 26863
Sentiment_Polarity        26863
Sentiment_Subjectivity    26863
dtype: int64

In [10]:
play_store_df = play_store_df[play_store_df["Installs"] != "Free"]
play_store_df["Installs_Num"] = (

    play_store_df["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
    .astype(float)

)

In [11]:
play_store_df[["Installs", "Installs_Num"]].head()

,Installs,Installs_Num
0,"10,000+",10000.0
1,"500,000+",500000.0
2,"5,000,000+",5000000.0
3,"50,000,000+",50000000.0
4,"100,000+",100000.0


In [12]:
play_store_df = play_store_df.dropna(subset=["Installs_Num"])

In [13]:
play_store_df["Installs_Num"].isnull().sum()

np.int64(0)

In [14]:
play_store_df.info()

<class 'pandas.DataFrame'>
Index: 10840 entries, 0 to 10840
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10840 non-null  str    
 1   Category        10840 non-null  str    
 2   Rating          9366 non-null   float64
 3   Reviews         10840 non-null  str    
 4   Size            10840 non-null  str    
 5   Installs        10840 non-null  str    
 6   Type            10839 non-null  str    
 7   Price           10840 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10840 non-null  str    
 10  Last Updated    10840 non-null  str    
 11  Current Ver     10832 non-null  str    
 12  Android Ver     10838 non-null  str    
 13  Installs_Num    10840 non-null  float64
dtypes: float64(2), str(12)
memory usage: 2.3 MB


In [15]:
merged_df = pd.merge(play_store_df, reviews_df, on='App', how='inner')

In [16]:
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Installs_Num,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,NaN,NaN,NaN,NaN
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,I love colors inspyering,Positive,0.500,0.600000


In [17]:
merged_df.shape

(122662, 18)

In [18]:
merged_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver', 'Installs_Num', 'Translated_Review', 'Sentiment',
       'Sentiment_Polarity', 'Sentiment_Subjectivity'],
      dtype='str')

In [20]:
merged_df["Category"].unique()

<ArrowStringArray>
[     'ART_AND_DESIGN',   'AUTO_AND_VEHICLES',              'BEAUTY',
 'BOOKS_AND_REFERENCE',            'BUSINESS',              'COMICS',
       'COMMUNICATION',              'DATING',           'EDUCATION',
       'ENTERTAINMENT',              'EVENTS',             'FINANCE',
      'FOOD_AND_DRINK',  'HEALTH_AND_FITNESS',      'HOUSE_AND_HOME',
  'LIBRARIES_AND_DEMO',           'LIFESTYLE',                'GAME',
              'FAMILY',             'MEDICAL',              'SOCIAL',
            'SHOPPING',         'PHOTOGRAPHY',              'SPORTS',
    'TRAVEL_AND_LOCAL',               'TOOLS',     'PERSONALIZATION',
        'PRODUCTIVITY',           'PARENTING',             'WEATHER',
       'VIDEO_PLAYERS',  'NEWS_AND_MAGAZINES', 'MAPS_AND_NAVIGATION']
Length: 33, dtype: str

In [21]:
filtered_df = merged_df[

    ~merged_df["Category"].str.startswith(
        ("A", "C", "G", "S"),
        na=False
    )

]

In [22]:
filtered_df["Category"].unique()

<ArrowStringArray>
[             'BEAUTY', 'BOOKS_AND_REFERENCE',            'BUSINESS',
              'DATING',           'EDUCATION',       'ENTERTAINMENT',
              'EVENTS',             'FINANCE',      'FOOD_AND_DRINK',
  'HEALTH_AND_FITNESS',      'HOUSE_AND_HOME',  'LIBRARIES_AND_DEMO',
           'LIFESTYLE',              'FAMILY',             'MEDICAL',
         'PHOTOGRAPHY',    'TRAVEL_AND_LOCAL',               'TOOLS',
     'PERSONALIZATION',        'PRODUCTIVITY',           'PARENTING',
             'WEATHER',       'VIDEO_PLAYERS',  'NEWS_AND_MAGAZINES',
 'MAPS_AND_NAVIGATION']
Length: 25, dtype: str

In [23]:
category_installs = filtered_df.groupby("Category")["Installs_Num"].sum()

In [24]:
category_installs = category_installs.reset_index()

In [25]:
category_installs.head()

,Category,Installs_Num
0,BEAUTY,6.731000e+08
1,BOOKS_AND_REFERENCE,8.194400e+10
2,BUSINESS,2.281600e+10
3,DATING,1.024380e+10
4,EDUCATION,1.100044e+11


In [26]:
filtered_df["Installs_Num"].head()

1151    500000.0
1152    500000.0
1153    500000.0
1154    500000.0
1155    500000.0
Name: Installs_Num, dtype: float64

In [27]:
filtered_df["Installs_Num"].dtype

dtype('float64')

In [28]:
filtered_df["Installs_Num"].max()

np.float64(1000000000.0)

In [29]:
print(play_store_df.shape)
print(reviews_df.shape)
print(merged_df.shape)

(10840, 14)
(64295, 5)
(122662, 18)


In [32]:
category_installs = filtered_df.groupby("Category")["Installs_Num"].sum()

In [33]:
category_installs = (
    filtered_df.groupby("Category")["Installs_Num"].sum().reset_index()
)

In [34]:
category_installs.head()

,Category,Installs_Num
0,BEAUTY,6.731000e+08
1,BOOKS_AND_REFERENCE,8.194400e+10
2,BUSINESS,2.281600e+10
3,DATING,1.024380e+10
4,EDUCATION,1.100044e+11


In [36]:
filtered_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Installs_Num,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
1151,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000.0,This a̸p̸p̸ Na Kare please Mere Bhai A L pleas...,Negative,-0.500000,0.950000
1152,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000.0,Worst lot add; time west,Negative,-1.000000,1.000000
1153,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000.0,bed bakvas time west stupid,Negative,-0.800000,1.000000
1154,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000.0,It bad dont install worst,Negative,-0.850000,0.833333
1155,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000.0,Fake install best sweet selfie,Positive,0.283333,0.650000


In [37]:
filtered_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver', 'Installs_Num', 'Translated_Review', 'Sentiment',
       'Sentiment_Polarity', 'Sentiment_Subjectivity'],
      dtype='str')

In [38]:
#remove duplicates
filtered_df = filtered_df.drop_duplicates(subset=["App"])
filtered_df.shape

(787, 18)

In [39]:
category_installs = filtered_df.groupby(
    "Category"
)["Installs_Num"].sum().reset_index()

In [40]:
category_installs

,Category,Installs_Num
0,BEAUTY,1.320000e+07
1,BOOKS_AND_REFERENCE,1.273200e+09
2,BUSINESS,1.733000e+08
3,DATING,7.414600e+07
4,EDUCATION,1.771100e+08
5,ENTERTAINMENT,1.271450e+09
6,EVENTS,2.410000e+06
7,FAMILY,1.902470e+09
8,FINANCE,2.076600e+08
9,FOOD_AND_DRINK,5.655000e+07


In [41]:
top5_categories = category_installs.sort_values(
    by="Installs_Num",
    ascending=False
).head(5)

In [42]:
top5_categories

,Category,Installs_Num
20,PRODUCTIVITY,3.705600e+09
21,TOOLS,3.182800e+09
7,FAMILY,1.902470e+09
19,PHOTOGRAPHY,1.819100e+09
16,NEWS_AND_MAGAZINES,1.617300e+09


In [43]:
top5_categories["Highlight"] = top5_categories["Installs_Num"] > 1000000
top5_categories

,Category,Installs_Num,Highlight
20,PRODUCTIVITY,3.705600e+09,True
21,TOOLS,3.182800e+09,True
7,FAMILY,1.902470e+09,True
19,PHOTOGRAPHY,1.819100e+09,True
16,NEWS_AND_MAGAZINES,1.617300e+09,True
